# Antenna Pattern Data Generation
This is the script used to convert the raw antenna pattern files to the text file used in LuPNT

In [ ]:
import pylupnt as pnt

### Antenna Pattern Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def create_antenna_plot(ax, eirp_data, azimuths, elevations, title):
    # Create a meshgrid for azimuth and elevation
    azimuths_rad = np.deg2rad(azimuths)
    elevations_rad = np.deg2rad(elevations)

    AZ, EL = np.meshgrid(azimuths_rad, elevations_rad)

    # Plot the EIRP data using pcolormesh
    # Note: pcolormesh expects the grid to be one larger in each dimension, so we need to define the edges
    # Calculate the edges for azimuth and elevation
    azimuths_edges = np.deg2rad(np.linspace(0, 360, len(azimuths) + 1))
    elevations_edges = np.deg2rad(np.linspace(0, 90, len(elevations) + 1))

    AZ_edges, EL_edges = np.meshgrid(azimuths_edges, elevations_edges)

    # Plot using pcolormesh
    c = ax.pcolormesh(AZ_edges, EL_edges, eirp_data, shading="auto", cmap="viridis")

    # Add a colorbar
    cb = plt.colorbar(c, ax=ax, pad=0.1)
    cb.set_label("Gain (dB)")  # Adjust units as necessary

    # Set the direction of the zero azimuth (boresight direction)
    boresight_deg = 0  # Change this value to set a different boresight direction
    ax.set_theta_zero_location("N")  # 'N' for north; adjust if needed
    ax.set_theta_direction(-1)  # Clockwise

    # Optionally, rotate the plot so that boresight_deg is at the top
    rotation_rad = np.deg2rad(boresight_deg)
    ax.set_theta_offset(-rotation_rad)

    # Set radial limits (elevation)
    ax.set_rlim(0, np.deg2rad(90))  # Assuming elevations up to 90°
    ax.set_rlabel_position(225)  # Position of the radial labels

    # Set labels
    ax.set_title("{}".format(title), va="bottom")

    # Customize ticks
    # Azimuth ticks
    az_ticks = np.deg2rad(np.arange(0, 360, 45))
    az_labels = [f"{angle}°" for angle in np.arange(0, 360, 45)]
    ax.set_xticks(az_ticks)
    ax.set_xticklabels(az_labels)

    # Elevation ticks
    el_ticks = np.deg2rad(np.arange(0, 91, 15))
    el_labels = [f"{angle}°" for angle in np.arange(0, 91, 15)]
    ax.set_yticks(el_ticks)
    ax.set_yticklabels(el_labels)

## GPS III Antenna Pattern
The files can be obtained from the Navigation Center Website 
https://www.navcen.uscg.gov/gps-technical-references

In [ ]:
from pypdf import PdfReader
import numpy as np

# Get Data Path
data_path = pnt.get_data_path()
gps_block3_raw = data_path + "/antenna/GPS/raw/GPS_III"
save_dir = data_path + "/antenna/GPS/txt"

svn = [74, 75, 76, 77, 78]
freq = ["L1", "L2", "L5"]

# svn = [74]
# freq = ["L1"]

# Figures
fig, axs = plt.subplots(5, 3, subplot_kw={"projection": "polar"}, figsize=(10, 15))

for i in range(len(svn)):
    for j in range(len(freq)):
        dir_name = (
            gps_block3_raw
            + "/GPS_III_SVN{}_EC_L1_2_5_Antenna_Patterns_Directivity".format(svn[i])
        )
        filename = (
            dir_name
            + "/GPS_III_SVN{}_EC_{}_Antenna_Patterns_Directivity.pdf".format(
                svn[i], freq[j]
            )
        )
        # load pdf file
        print("Loading file: {}".format(filename))

        reader = PdfReader(filename)

        # creating a page object
        page = reader.pages[0]

        # extracting text from page
        text = page.extract_text()

        # Extract the 5 th line
        lines = text.split("\n")

        phi = np.arange(0, 360, 10)
        # insert -50 to the first element
        phi = np.insert(phi, 0, -50)

        # Create new text, where the
        # the first row is [-50, 0, 10, ..., 350]
        # from the second row, the text is the same as the original text lines 4 to the end
        new_text = " ".join([str(x) for x in phi]) + "\n" + "\n".join(lines[4:])

        # add "," between the elements
        new_text = new_text.replace(" ", ",")
        # print(new_text)

        # save the new text to a file
        if freq[j] == "L1":
            save_file = save_dir + "/SVN{}_LM.txt".format(svn[i], freq[j])
            with open(save_file, "w") as f:
                f.write(new_text)

        save_file = save_dir + "/SVN{}_LM_{}.txt".format(svn[i], freq[j])

        with open(save_file, "w") as f:
            f.write(new_text)

        print("Save to file: {}".format(save_file))

        # load text file as numpy array
        data = np.loadtxt(save_file, delimiter=",")

        # Extract the azimuths and elevations
        azimuths = data[0, 1:]
        elevations = data[1:, 0]
        vals = data[1:, 1:]

        # extract the data where elevation > 0
        vals = vals[elevations > 0, :]
        elevations = elevations[elevations > 0]

        # Create a figure and axis
        ax = axs[i, j]
        create_antenna_plot(
            ax, vals, azimuths, elevations, "SVN{} {}".format(svn[i], freq[j])
        )

plt.tight_layout()
plt.show()

## GALILEO Antenna Pattern
Galileo antenna patterns can be obtained from the EU Science Hub Website
https://joint-research-centre.ec.europa.eu/scientific-activities-z/galileo-reference-antenna-pattern-grap-model_en

Note that the given values are EIRP values, and not antenna gains.
Therefore, we assume a fixed transmission power of 14 dBW, and subtract that from the EIRP to obtain the gain

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

grap_raw = data_path + "/antenna/Galileo/raw/"
save_dir = data_path + "/antenna/Galileo/txt"

types = ["E1", "E5a", "E5b", "E6"]
types_str = ["E1__", "E5a_", "E5b_", "E6__"]

trans_power = 14.0

fig, axs = plt.subplots(2, 2, subplot_kw={"projection": "polar"}, figsize=(8, 8))

for i in range(len(types)):
    filename = grap_raw + "GRAP_File_{}.xlsx".format(types_str[i])
    # load pdf file
    print("Loading file: {}".format(filename))

    df = pd.read_excel(filename, sheet_name="GRAP_EIRP_dBW_{}".format(types_str[i]))

    # second column is the azimuth
    az = df.iloc[:, 1].values[1:].astype(np.float32)  # (361,)

    # second row is the coelevation
    coe = df.iloc[0, :].values[2:].astype(np.float32)  # (91,)

    # vals
    vals = df.iloc[1:, 2:].values.astype(np.float32)  # (361, 91)
    vals = vals - trans_power

    # Create new text, where the
    # [0, 1:] is [-50, 0, 1, ..., 361] (Azimuth)
    # [1:, 0] is [0, 1, ..., 90] (Coelevation)
    # [0, 0] is -50
    # [1:, 1:] is vals.T
    data_mat = np.zeros((92, 362))
    data_mat[0, 1:] = az  # to float
    data_mat[1:, 0] = coe
    data_mat[0, 0] = -50
    data_mat[1:, 1:] = vals.T

    # save the new text to a file
    save_file = save_dir + "/Galileo_{}.txt".format(types[i])
    np.savetxt(save_file, data_mat, delimiter=",", fmt="%f")
    print("Save to file: {}".format(save_file))

    # plot antenna pattern as a polor color map
    row = i // 2
    col = i % 2
    create_antenna_plot(axs[row][col], vals.T, az, coe, "Galileo {}".format(types[i]))

plt.tight_layout()
plt.show()

## QZSS Antenna Pattern
QZSS antenna patterns can be obtained from the QZSS Website
https://qzss.go.jp/en/technical/antenna-patterns.html

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

qzs_raw = data_path + "/antenna/QZSS/raw/"
save_dir = data_path + "/antenna/QZSS/txt"

types_str = ["1R", "02", "03", "04", "05", "06", "07"]
xlsx_type = [1, 2, 1, 2, 3, 3, 3, 3]
freq = ["L1", "L2", "L5"]

fig, axs = plt.subplots(
    len(types_str), len(freq), subplot_kw={"projection": "polar"}, figsize=(8, 16)
)

data_type_a = []

for i in range(len(types_str)):
    for j in range(len(freq)):
        if (xlsx_type[i] == 3) and (j == 1):
            # no data for L2 for 05, 06, 07
            continue

        filename = qzs_raw + "QZS{}_ANT_Pattern.xlsx".format(types_str[i])
        # load pdf file
        print("Loading file: {}".format(filename))

        df = pd.read_excel(filename, sheet_name="{}".format(freq[j]))

        if xlsx_type[i] == 1:
            az = np.arange(0, 360, 30).astype(np.float32)
            el = np.arange(0, 61, 1).astype(np.float32)
            val_row0 = 3
            val_col0 = 1
        elif xlsx_type[i] == 2:
            az = np.arange(0, 360, 30).astype(np.float32)
            el = np.arange(0, 90.5, 0.5).astype(np.float32)
            val_row0 = 4
            val_col0 = 2

        elif xlsx_type[i] == 3:
            az = np.arange(0, 360, 10).astype(np.float32)
            el = np.arange(0, 90.1, 0.1).astype(np.float32)
            val_row0 = 3
            val_col0 = 1

        # print("az: ", az)
        # print("el: ", el)

        # construct data
        vals = df.iloc[val_row0:, val_col0:].values.astype(np.float32)
        # print("vals: ", vals)

        # for xlsx type 1, extend the el values to 90 by linear extrapolation
        if xlsx_type[i] == 1:
            data_mat = np.zeros((92, 13))
            data_mat[0, 1:] = az
            data_mat[1:, 0] = np.arange(0, 91, 1)
            data_mat[0, 0] = -50

            # linear extrapolation
            for k in range(1, 13):
                ref = np.arange(0, 91, 1).astype(np.float32)
                data_mat[1:, k] = np.interp(ref, el, vals[:, k - 1])

            el = np.arange(0, 91, 1).astype(np.float32)
            vals = data_mat[1:, 1:]
        else:
            data_mat = np.zeros((el.shape[0] + 1, az.shape[0] + 1))
            data_mat[0, 1:] = az  # to float
            data_mat[1:, 0] = el
            data_mat[0, 0] = -50
            data_mat[1:, 1:] = vals

        # save the new text to a file
        save_file = save_dir + "/QZSS_{}_{}.txt".format(types_str[i], freq[j])
        np.savetxt(save_file, data_mat, delimiter=",", fmt="%f")
        print("Save to file: {}".format(save_file))

        # plot antenna pattern as a polor color map
        create_antenna_plot(
            axs[i][j], vals, az, el, "Gain QZSS {} {}".format(types_str[i], freq[j])
        )

plt.tight_layout()
plt.show()